In [1]:
!test -f obs_3day.bufr \
    || wget https://sites.ecmwf.int/repository/pdbufr/test-data/obs_3day.bufr \
             --output-document=obs_3day.bufr

# Flat reader: `required_columns` – individual key extraction mode

In [2]:
import pdbufr

## Default behaviour (`required_columns=True`)

The default value is `True`, which in individual key extraction mode means that **all** keys listed in `columns` must be present in a message for it to be included.  Messages where any column is absent are silently skipped, and the result is an empty DataFrame if no message satisfies the requirement.

In `obs_3day.bufr` no message contains both precipitation keys, so requesting both together with the default `required_columns=True` returns 0 rows:

In [3]:
df_default = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "#1#totalPrecipitationPast6Hours", "#1#totalPrecipitationPast24Hours"],
    reader="flat",
)
print(f"{len(df_default)} rows")
# No message contains both keys, so the DataFrame is empty.
# Use required_columns=False to include all messages and fill
# absent keys with NaN instead.
try:
    df_default[["ident", "#1#totalPrecipitationPast6Hours", "#1#totalPrecipitationPast24Hours"]].head()
except KeyError:
    print("DataFrame is empty – no message contains both precipitation keys.")
    print("Pass required_columns=False to allow NaN for absent keys.")

0 rows
DataFrame is empty – no message contains both precipitation keys.
Pass required_columns=False to allow NaN for absent keys.


## Disabling `required_columns` (`required_columns=False`)

Setting `required_columns=False` in individual key extraction mode allows any of the ``columns`` to be missing.

In [4]:
df_false = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "#1#totalPrecipitationPast6Hours", "#1#totalPrecipitationPast24Hours"],
    required_columns=False,
    reader="flat",
)
print(f"{len(df_false)} rows")

50 rows


In [5]:
df_false.head()

,ident,latitude,#1#totalPrecipitationPast6Hours,#1#totalPrecipitationPast24Hours
0,03894,49.43,0.0,None
1,03590,52.12,0.0,None
2,03379,53.03,0.0,None
3,03391,53.09,0.0,None
4,03743,51.20,0.0,None


## Skipping messages that lack a key

Setting `required_columns` to a key name (or a list of key names) causes messages that do not contain **all** of those keys to be skipped.

In [6]:
# Only messages that report 6-hour precipitation
df_6h = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "#1#totalPrecipitationPast6Hours"],
    required_columns=["#1#totalPrecipitationPast6Hours"],
    reader="flat",
)
print(f"{len(df_6h)} rows (messages with 6-hour precip)")
df_6h.head()

43 rows (messages with 6-hour precip)


,ident,latitude,#1#totalPrecipitationPast6Hours
0,03894,49.43,0.0
1,03590,52.12,0.0
2,03379,53.03,0.0
3,03391,53.09,0.0
4,03743,51.20,0.0


In [7]:
# Only messages that report 24-hour precipitation
df_24h = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "#1#totalPrecipitationPast24Hours"],
    required_columns=["#1#totalPrecipitationPast24Hours"],
    reader="flat",
)
print(f"{len(df_24h)} rows (messages with 24-hour precip)")
df_24h.head()

7 rows (messages with 24-hour precip)


,ident,latitude,#1#totalPrecipitationPast24Hours
0,03105,55.68,None
1,01400,56.54,None
2,01300,61.20,None
3,06252,53.22,None
4,06310,51.44,None


## Requiring a key that does not exist

If a key listed in `required_columns` is absent from **all** messages, the result is an empty DataFrame:

In [8]:
df_empty = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude"],
    required_columns=["nonExistentKey"],
    reader="flat",
)
print(f"rows: {len(df_empty)}, empty: {df_empty.empty}")

rows: 0, empty: True


## Rank-matching in `required_columns`

Keys listed in `required_columns` are matched against **any** rank.  Specifying `"totalPrecipitationPast6Hours"` (no rank) is equivalent to requiring the key to be present with any rank in the message.

In [9]:
# Equivalent to the ranked example above
df_any_rank = pdbufr.read_bufr(
    "obs_3day.bufr",
    columns=["ident", "latitude", "#1#totalPrecipitationPast6Hours"],
    required_columns=["totalPrecipitationPast6Hours"],
    reader="flat",
)
print(f"{len(df_any_rank)} rows")
assert len(df_any_rank) == len(df_6h)

43 rows
